# Learning Rate Experiments for Vanilla DQN

This notebook runs a learning-rate ablation study for the vanilla DQN LunarLander baseline. All baseline parameters are kept fixed except `learning_rate`.


## 1. Setup and Imports


In [1]:
import sys
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from core.vanilla_dqn_lunar_lander import (
    train_dqn,
    evaluate_dqn,
    save_plot,
    save_results,
    save_model,
)


Using device: cpu


## 2. Experiment Settings

Keep the vanilla baseline settings unchanged and vary only the learning rate.


In [ ]:
learning_rates = [3e-3, 2e-3, 1e-3, 8e-4, 7e-4, 5e-4, 3e-4, 1e-4, 5e-5]
num_episodes = 500
num_eval_episodes = 10

# ======== Multi-seed experiment setup ========
# Use the same five seeds for every learning rate so the comparison is fairer.
seeds = [0, 1, 2, 3, 4]
# ======== End multi-seed experiment setup ========

base_hyperparameters = {
    'batch_size': 32,
    'update_frequency': 4,
    'target_update_frequency': 100,
    'num_layers': 2,
    'hidden_dim': 128,
    'epsilon_decay': 0.995,
    'gamma': 0.99,
    'num_episodes': num_episodes,
    'num_eval_episodes': num_eval_episodes,
    'seeds': seeds,
}


## 3. Run Learning Rate Ablation

Each learning rate is trained, evaluated, and saved automatically under `results/ablations/learning_rate/<learning_rate>/`.


In [ ]:
all_results = {}
run_records = []

print('=' * 70)
print('DQN Learning Rate Ablation Study')
print(f'Testing {len(learning_rates)} learning rates x {len(seeds)} seeds = {len(learning_rates) * len(seeds)} training runs')
print('=' * 70)

for lr in learning_rates:
    lr_variant_name = f'{lr:.0e}'
    all_results[lr_variant_name] = {
        'learning_rate': lr,
        'runs': [],
    }

    for seed in seeds:
        # ======== Multi-seed learning-rate run ========
        # Each learning rate is trained with the same seed set to reduce lucky-run bias.
        variant_name = f'{lr_variant_name}_seed_{seed}'

        print('' + '=' * 70)
        print(f'Training with learning_rate = {lr} ({lr_variant_name}), seed = {seed}')
        print('=' * 70)

        agent, rewards, losses = train_dqn(
            num_episodes=num_episodes,
            batch_size=base_hyperparameters['batch_size'],
            update_frequency=base_hyperparameters['update_frequency'],
            target_update_frequency=base_hyperparameters['target_update_frequency'],
            num_layers=base_hyperparameters['num_layers'],
            hidden_dim=base_hyperparameters['hidden_dim'],
            epsilon_decay=base_hyperparameters['epsilon_decay'],
            learning_rate=lr,
            seed=seed,
        )

        print('Evaluating trained agent...')
        eval_rewards = evaluate_dqn(agent, num_episodes=num_eval_episodes, seed=seed)

        hyperparameters = dict(base_hyperparameters)
        hyperparameters['learning_rate'] = lr
        hyperparameters['seed'] = seed

        save_results(
            agent,
            rewards,
            losses,
            hyperparameters,
            experiment_name='learning_rate',
            variant_name=variant_name,
        )
        save_model(agent, experiment_name='learning_rate', variant_name=variant_name)
        save_plot(rewards, losses, experiment_name='learning_rate', variant_name=variant_name)

        final_avg_reward = float(np.mean(rewards[-25:]))
        eval_avg_reward = float(np.mean(eval_rewards))
        best_reward = float(max(rewards))

        run_result = {
            'seed': seed,
            'train_rewards': rewards,
            'losses': losses,
            'eval_rewards': eval_rewards,
            'final_avg_reward': final_avg_reward,
            'eval_avg_reward': eval_avg_reward,
            'best_reward': best_reward,
        }
        all_results[lr_variant_name]['runs'].append(run_result)

        run_records.append({
            'variant': lr_variant_name,
            'learning_rate': lr,
            'seed': seed,
            'final_avg_reward': final_avg_reward,
            'eval_avg_reward': eval_avg_reward,
            'best_training_reward': best_reward,
        })

        print(f'Finished {variant_name}: final train avg={final_avg_reward:.2f}, eval avg={eval_avg_reward:.2f}')
        # ======== End multi-seed learning-rate run ========

    # ======== Aggregate learning-rate results across seeds ========
    runs = all_results[lr_variant_name]['runs']
    all_results[lr_variant_name]['train_rewards'] = np.mean([run['train_rewards'] for run in runs], axis=0)
    all_results[lr_variant_name]['final_avg_reward'] = float(np.mean([run['final_avg_reward'] for run in runs]))
    all_results[lr_variant_name]['final_avg_reward_std'] = float(np.std([run['final_avg_reward'] for run in runs]))
    all_results[lr_variant_name]['eval_avg_reward'] = float(np.mean([run['eval_avg_reward'] for run in runs]))
    all_results[lr_variant_name]['eval_avg_reward_std'] = float(np.std([run['eval_avg_reward'] for run in runs]))
    all_results[lr_variant_name]['best_reward'] = float(np.mean([run['best_reward'] for run in runs]))
    all_results[lr_variant_name]['best_reward_std'] = float(np.std([run['best_reward'] for run in runs]))
    # ======== End aggregate learning-rate results across seeds ========

print('' + '=' * 70)
print('Learning rate ablation complete')
print('=' * 70)


## 4. Compare Results


In [ ]:
summary_rows = []

for variant_name, result in all_results.items():
    summary_rows.append({
        'variant': variant_name,
        'learning_rate': result['learning_rate'],
        'final_avg_reward_last_25_mean': result['final_avg_reward'],
        'final_avg_reward_last_25_std': result['final_avg_reward_std'],
        'eval_avg_reward_mean': result['eval_avg_reward'],
        'eval_avg_reward_std': result['eval_avg_reward_std'],
        'best_training_reward_mean': result['best_reward'],
        'best_training_reward_std': result['best_reward_std'],
    })

summary_rows = sorted(summary_rows, key=lambda row: row['eval_avg_reward_mean'], reverse=True)

# ======== Multi-seed summary table ========
print('Learning Rate Experiment Summary Across Seeds')
print('-' * 125)
print(f"{'Variant':<10} {'LR':<12} {'Final Avg (25)':<24} {'Eval Avg':<24} {'Best Train':<24}")
print('-' * 125)
for row in summary_rows:
    print(
        f"{row['variant']:<10} "
        f"{row['learning_rate']:<12.1e} "
        f"{row['final_avg_reward_last_25_mean']:>8.2f} +/- {row['final_avg_reward_last_25_std']:<8.2f} "
        f"{row['eval_avg_reward_mean']:>8.2f} +/- {row['eval_avg_reward_std']:<8.2f} "
        f"{row['best_training_reward_mean']:>8.2f} +/- {row['best_training_reward_std']:<8.2f}"
    )
# ======== End multi-seed summary table ========


## 5. Plot Comparison


In [ ]:
plt.figure(figsize=(12, 6))

for variant_name, result in all_results.items():
    # ======== Plot mean training curve across seeds ========
    rewards = result['train_rewards']
    window = min(50, max(1, len(rewards) // 2))
    if window > 1:
        smoothed = np.convolve(rewards, np.ones(window) / window, mode='valid')
        x_values = range(window - 1, window - 1 + len(smoothed))
        plt.plot(x_values, smoothed, label=f'LR {variant_name}')
    else:
        plt.plot(rewards, label=f'LR {variant_name}')
    # ======== End plot mean training curve across seeds ========

plt.axhline(y=200, color='black', linestyle='--', linewidth=1, alpha=0.6, label='Target 200')
plt.xlabel('Episode')
plt.ylabel('Mean Smoothed Reward Across Seeds')
plt.title('Learning Rate Comparison for Vanilla DQN Across Seeds')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Summary and Analysis

This experiment tested nine learning rates for the vanilla DQN agent while keeping all other baseline settings fixed: 500 training episodes, batch size 32, two hidden layers, hidden dimension 128, epsilon decay 0.995, and gamma 0.99. The tested learning rates were `3e-3 (0.003)`, `2e-3 (0.002)`, `1e-3 (0.001)`, `8e-4 (0.0008)`, `7e-4 (0.0007)`, `5e-4 (0.0005)`, `3e-4 (0.0003)`, `1e-4 (0.0001)`, and `5e-5 (0.00005)`.

### Summary Statistics

The table below compares three complementary indicators: the final training average over the last 25 episodes, the average evaluation reward, and the best single training reward. As with the epsilon-decay study, the most useful interpretation comes from comparing these metrics together rather than selecting the largest number in one column.

| Learning Rate | Final Avg Reward (Last 25) | Evaluation Avg Reward | Best Training Reward |
|---:|---:|---:|---:|
| `5e-4 (0.0005)` | `19.99` | `98.22` | `282.60` |
| `2e-3 (0.002)` | `-45.91` | `64.22` | `241.65` |
| `1e-3 (0.001)` | `-25.04` | `53.58` | `294.79` |
| `1e-4 (0.0001)` | `16.47` | `40.43` | `299.72` |
| `7e-4 (0.0007)` | `-18.37` | `25.96` | `274.04` |
| `5e-5 (0.00005)` | `13.89` | `9.03` | `262.33` |
| `8e-4 (0.0008)` | `-2.88` | `6.43` | `275.12` |
| `3e-3 (0.003)` | `18.92` | `-145.30` | `306.58` |
| `3e-4 (0.0003)` | `-39.54` | `-179.05` | `295.49` |

A high best training reward does not necessarily indicate a reliable policy. For example, `3e-3 (0.003)` achieved the highest single training reward (`306.58`), but its evaluation average was strongly negative (`-145.30`). This suggests that the updates were too aggressive: the agent occasionally found a strong trajectory, but the learned policy was unstable and did not transfer reliably to evaluation episodes. Similarly, `3e-4 (0.0003)` reached a high best training reward (`295.49`) but had the worst evaluation average (`-179.05`), showing that isolated peaks can be misleading in a noisy reinforcement-learning setting.

### Training Curve Interpretation

**Unstable High-Learning-Rate Behavior (`3e-3 (0.003)`)**

The largest learning rate produced sharp learning jumps and the highest best reward, but it failed during evaluation. This pattern is consistent with unstable Q-value updates: large gradient steps can rapidly exploit recent experience, but they can also overshoot useful value estimates and produce a brittle policy. The positive final training average is therefore not enough evidence of robust learning.

**Strong and Reliable Mid-Range Learning (`5e-4 (0.0005)`, `1e-3 (0.001)`, `2e-3 (0.002)`)**

The middle range produced the most useful evaluation performance. `5e-4 (0.0005)` was clearly the strongest overall, with the highest evaluation average (`98.22`) and a positive final training average (`19.99`). `1e-3 (0.001)` and `2e-3 (0.002)` also achieved positive evaluation averages, but their final training averages remained negative, suggesting that their policies were less stable near the end of training. Among these, `5e-4 (0.0005)` offered the best balance between learning speed and update stability.

**Slow or Inconsistent Low-Learning-Rate Behavior (`1e-4 (0.0001)`, `5e-5 (0.00005)`)**

The smallest learning rates were more conservative. `1e-4 (0.0001)` and `5e-5 (0.00005)` both ended with positive final training averages, but their evaluation averages were much lower than `5e-4 (0.0005)`. This suggests that the agent was learning, but the updates were too small to consistently refine a strong policy within only 500 episodes. These learning rates may benefit from longer training, but they are less efficient under the current experiment budget.

**Misleading Middle-Low Result (`3e-4 (0.0003)`, `8e-4 (0.0008)`, `7e-4 (0.0007)`)**

The `3e-4 (0.0003)`, `8e-4 (0.0008)`, and `7e-4 (0.0007)` settings produced mixed or weak evaluation results despite reaching respectable best training rewards. This again highlights the gap between occasional success and reliable policy quality. In particular, `3e-4 (0.0003)` is a clear example of a learning rate that can produce good individual episodes without converging to a dependable final policy.

### Final Performance vs. Evaluation Performance

The comparison between final training average and evaluation average is especially important. `3e-3 (0.003)`, `1e-4 (0.0001)`, and `5e-5 (0.00005)` all ended with positive final training averages, but only `1e-4 (0.0001)` produced a moderately positive evaluation score, and even that was far below `5e-4 (0.0005)`. Conversely, `2e-3 (0.002)` had a negative final training average but a reasonably strong evaluation average (`64.22`). This mismatch suggests that the last 25 training episodes alone are not a sufficient criterion for selecting the learning rate.

Evaluation performance is the more reliable selection metric here because it measures the learned greedy policy after training, rather than the noisy training process that still includes exploration. Based on this criterion, `5e-4 (0.0005)` is the strongest learning rate by a large margin.

### Conclusion

Overall, the learning-rate ablation shows a clear trade-off. Very large learning rates can create unstable policies, while very small learning rates may learn too slowly within the available 500 episodes. The best-performing region is the mid-range, especially `5e-4 (0.0005)`, which achieved the highest evaluation reward and maintained a positive final training average. Therefore, this experiment recommends `learning_rate = 5e-4 (0.0005)` for the vanilla DQN configuration.

